# 04 — Multi-Label Disease Risk Classification

**Labels:** Anaemia, Diabetes_Risk, Dyslipidemia, Kidney_Risk, Liver_Stress, Thyroid_Abnormal  
**Label source:** `data/processed/train_labels.csv` / `test_labels.csv` (generated by 03c_ClusterRangeLearning)  
**Training:** Per-label binary relevance with biomarker exclusion + validation-set threshold tuning.

In [1]:
import os, warnings, joblib
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    f1_score, precision_score, recall_score,
    roc_auc_score, average_precision_score,
    hamming_loss, accuracy_score,
    confusion_matrix, ConfusionMatrixDisplay,
    precision_recall_curve
)

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', font_scale=1.0)
plt.rcParams['figure.dpi'] = 120

PROCESSED = '../data/processed'
MODELS    = '../models'
PLOTS     = '../results/plots'
METRICS   = '../results/metrics'
for d in [MODELS, PLOTS, METRICS]:
    os.makedirs(d, exist_ok=True)

LABELS = ['Anaemia','Diabetes_Risk','Dyslipidemia','Kidney_Risk','Liver_Stress','Thyroid_Abnormal']
print('Ready.')

Ready.


## Step 1 — Load Data

In [2]:
# Load scaled/unscaled data and cluster assignments
train_scaled = pd.read_csv(f'{PROCESSED}/train_wide_scaled.csv')
test_scaled  = pd.read_csv(f'{PROCESSED}/test_wide_scaled.csv')
train_clust  = pd.read_csv(f'{PROCESSED}/train_clusters.csv')
test_clust   = pd.read_csv(f'{PROCESSED}/test_clusters.csv')

# Merge cluster_id into scaled frames
train_scaled = train_scaled.merge(train_clust, on='document_id', how='left')
test_scaled  = test_scaled.merge(test_clust,  on='document_id', how='left')

print(f'Train (scaled): {train_scaled.shape}')
print(f'Test  (scaled): {test_scaled.shape}')
print(f'Clusters — train unique: {train_scaled["cluster_id"].nunique()}')

# FIX D: split train into fit-set and validation-set for threshold tuning
train_doc_ids, val_doc_ids = train_test_split(
    train_scaled['document_id'],
    test_size=0.15,
    random_state=42
)
train_fit = train_scaled[train_scaled['document_id'].isin(train_doc_ids)].reset_index(drop=True)
train_val = train_scaled[train_scaled['document_id'].isin(val_doc_ids)].reset_index(drop=True)
print(f'Fit rows: {len(train_fit):,}  |  Val rows: {len(train_val):,}')

Train (scaled): (79993, 110)
Test  (scaled): (19999, 110)
Clusters — train unique: 5
Fit rows: 67,994  |  Val rows: 11,999


## Step 2 — Load Labels

Labels were generated by `03c_ClusterRangeLearning.ipynb` using cluster-specific clinical thresholds.  
**Do not regenerate labels here.**

In [3]:
# FIX B: load labels from file — never generate here
train_label_path = f'{PROCESSED}/train_labels.csv'
test_label_path  = f'{PROCESSED}/test_labels.csv'

if not os.path.exists(train_label_path) or not os.path.exists(test_label_path):
    raise FileNotFoundError(
        'Run 03c_ClusterRangeLearning.ipynb first to generate '
        'train_labels.csv and test_labels.csv.'
    )

train_labels = pd.read_csv(train_label_path).set_index('document_id')
test_labels  = pd.read_csv(test_label_path).set_index('document_id')
print(f'Train labels: {train_labels.shape}  |  Test labels: {test_labels.shape}')
print('Labels:', list(train_labels.columns))

Train labels: (79993, 6)  |  Test labels: (19999, 6)
Labels: ['Anaemia', 'Diabetes_Risk', 'Dyslipidemia', 'Kidney_Risk', 'Liver_Stress', 'Thyroid_Abnormal']


In [4]:
print('=== Class Imbalance (Train) ===')
for lbl in LABELS:
    if lbl in train_labels.columns:
        pos_pct = train_labels[lbl].mean() * 100
        print(f'  {lbl:<20} {pos_pct:.1f}% positive')
    else:
        print(f'  {lbl:<20} *** column not found in labels file ***')

=== Class Imbalance (Train) ===
  Anaemia              15.4% positive
  Diabetes_Risk        8.2% positive
  Dyslipidemia         20.7% positive
  Kidney_Risk          19.2% positive
  Liver_Stress         26.8% positive
  Thyroid_Abnormal     24.6% positive


## Step 3 — Feature Engineering (Per-Label Biomarker Exclusion)

Models are trained **per label** using binary relevance.  
Direct biomarker columns that define each label are excluded (case-insensitive substring match).  
`cluster_id` is kept as a feature.

In [5]:
# FIX C: updated exclusion dict + case-insensitive substring helper
LABEL_EXCLUDE = {
    'Anaemia':          ['haemoglobin','hemoglobin','hgb','mcv','mch','mchc','rbc','rdw',
                         'pcv','hematocrit','reticulocyte'],
    'Diabetes_Risk':    ['hba1c','hemoglobin a1c','glycated','glucose','fasting glucose',
                         'pp glucose','blood sugar'],
    'Dyslipidemia':     ['total cholesterol','cholesterol','ldl','hdl','triglyceride',
                         'triglycerides','vldl','lipoprotein'],
    'Kidney_Risk':      ['creatinine','bun','blood urea nitrogen','urea','egfr','gfr','uric acid'],
    'Liver_Stress':     ['alt','sgpt','ast','sgot','ggt','bilirubin','albumin','alp',
                         'alkaline phosphatase','alk phos'],
    'Thyroid_Abnormal': ['tsh','t3','t4','ft3','ft4','thyroid','triiodothyronine','thyroxine'],
}

def feature_cols_for_label(df, label):
    base_exclude = {'document_id', 'age', 'gender'}
    blocked_terms = LABEL_EXCLUDE[label]
    cols = []
    for c in df.columns:
        c_lower = c.lower()
        if c in base_exclude:
            continue
        if any(term.lower() in c_lower for term in blocked_terms):
            continue
        cols.append(c)
    return cols

# Sanity check
for lbl in LABELS:
    cols = feature_cols_for_label(train_scaled, lbl)
    print(f'{lbl:<20}: {len(cols)} features')

Anaemia             : 98 features
Diabetes_Risk       : 101 features
Dyslipidemia        : 97 features
Kidney_Risk         : 95 features
Liver_Stress        : 91 features
Thyroid_Abnormal    : 101 features


In [6]:
# Align label arrays to fit/val/test document_id order
def get_y(df_scaled, lbl_df):
    return lbl_df[LABELS].reindex(df_scaled['document_id']).values

y_fit  = get_y(train_fit,  train_labels)
y_val  = get_y(train_val,  train_labels)
y_test = get_y(test_scaled, test_labels)
print(f'y_fit: {y_fit.shape}  y_val: {y_val.shape}  y_test: {y_test.shape}')

y_fit: (67994, 6)  y_val: (11999, 6)  y_test: (19999, 6)


## Step 4 — Train Models (Per-Label Binary Relevance)

Each model is trained separately per label on label-specific non-leaking features.  
Fit set: 85% of train. Validation set: 15% of train (used only for threshold tuning).

In [7]:
# FIX C+D: per-label binary relevance training on fit set only
from sklearn.base import clone

BASE_MODELS = {
    'RF_base':     RandomForestClassifier(n_estimators=200, n_jobs=-1, random_state=42),
    'RF_balanced': RandomForestClassifier(n_estimators=200, class_weight='balanced',
                                          n_jobs=-1, random_state=42),
    'KNN':         KNeighborsClassifier(n_neighbors=5, n_jobs=-1),
    'SVM':         CalibratedClassifierCV(LinearSVC(class_weight='balanced', max_iter=2000)),
}

# trained_models[model_name][label] = fitted estimator
trained_models = {name: {} for name in BASE_MODELS}

for name, base_est in BASE_MODELS.items():
    print(f'Training {name}...')
    for i, lbl in enumerate(LABELS):
        cols = feature_cols_for_label(train_fit, lbl)
        X_tr_lbl = train_fit[cols].values
        y_tr_lbl = y_fit[:, i]
        est = clone(base_est)
        est.fit(X_tr_lbl, y_tr_lbl)
        trained_models[name][lbl] = est
    joblib.dump(trained_models[name], f'{MODELS}/{name.lower()}.pkl')
    print(f'  {name} saved ({len(LABELS)} label estimators).')
print('All models trained.')

Training RF_base...
  RF_base saved (6 label estimators).
Training RF_balanced...
  RF_balanced saved (6 label estimators).
Training KNN...
  KNN saved (6 label estimators).
Training SVM...
  SVM saved (6 label estimators).
All models trained.


## Step 5 — Per-Label Threshold Tuning (Validation Set)

Thresholds are tuned on the **validation set** (15% of train) to avoid leakage.  
Final test evaluation uses these fixed thresholds.

In [8]:
# FIX D: tune thresholds on validation set
THRESHOLDS = np.arange(0.1, 0.95, 0.05)

def tune_per_label(model_dict, val_df, y_val_arr, labels):
    """Tune threshold per label on validation set. Returns dict: label -> best_threshold."""
    thresholds = {}
    for i, lbl in enumerate(labels):
        est = model_dict[lbl]
        cols = feature_cols_for_label(val_df, lbl)
        X_val_lbl = val_df[cols].values
        try:
            p = est.predict_proba(X_val_lbl)[:, 1]
        except AttributeError:
            thresholds[lbl] = 0.5
            continue
        best_f1, best_t = 0.0, 0.5
        for t in THRESHOLDS:
            f1 = f1_score(y_val_arr[:, i], (p >= t).astype(int), zero_division=0)
            if f1 > best_f1:
                best_f1, best_t = f1, t
        thresholds[lbl] = round(best_t, 2)
    return thresholds

model_thresholds = {}
for name, model_dict in trained_models.items():
    model_thresholds[name] = tune_per_label(model_dict, train_val, y_val, LABELS)
    print(f'{name}: {model_thresholds[name]}')

thr_df = pd.DataFrame(model_thresholds, index=LABELS)
print('\nOptimal thresholds (tuned on val set):')
print(thr_df.to_string())

RF_base: {'Anaemia': np.float64(0.25), 'Diabetes_Risk': np.float64(0.2), 'Dyslipidemia': np.float64(0.25), 'Kidney_Risk': np.float64(0.25), 'Liver_Stress': np.float64(0.3), 'Thyroid_Abnormal': np.float64(0.3)}
RF_balanced: {'Anaemia': np.float64(0.2), 'Diabetes_Risk': np.float64(0.15), 'Dyslipidemia': np.float64(0.2), 'Kidney_Risk': np.float64(0.2), 'Liver_Stress': np.float64(0.3), 'Thyroid_Abnormal': np.float64(0.25)}
KNN: {'Anaemia': np.float64(0.1), 'Diabetes_Risk': np.float64(0.2), 'Dyslipidemia': np.float64(0.1), 'Kidney_Risk': np.float64(0.1), 'Liver_Stress': np.float64(0.1), 'Thyroid_Abnormal': np.float64(0.1)}
SVM: {'Anaemia': np.float64(0.2), 'Diabetes_Risk': np.float64(0.15), 'Dyslipidemia': np.float64(0.2), 'Kidney_Risk': np.float64(0.2), 'Liver_Stress': np.float64(0.25), 'Thyroid_Abnormal': np.float64(0.2)}

Optimal thresholds (tuned on val set):
                  RF_base  RF_balanced  KNN   SVM
Anaemia              0.25         0.20  0.1  0.20
Diabetes_Risk        0.20    

## Step 6 — Evaluation on Test Set

In [9]:
# FIX C: per-label evaluation using label-specific feature columns
def eval_per_label_binary(model_dict, test_df, y_test_arr, thresholds, labels):
    rows = []
    preds_all = np.zeros((len(test_df), len(labels)), dtype=int)
    proba_store = {}
    for i, lbl in enumerate(labels):
        est  = model_dict[lbl]
        cols = feature_cols_for_label(test_df, lbl)
        X_te_lbl = test_df[cols].values
        try:
            p = est.predict_proba(X_te_lbl)[:, 1]
        except AttributeError:
            p = est.predict(X_te_lbl).astype(float)
        pred = (p >= thresholds[lbl]).astype(int)
        preds_all[:, i] = pred
        proba_store[lbl] = p
        rows.append({
            'label':     lbl,
            'F1':        f1_score(y_test_arr[:, i], pred, zero_division=0),
            'Precision': precision_score(y_test_arr[:, i], pred, zero_division=0),
            'Recall':    recall_score(y_test_arr[:, i], pred, zero_division=0),
            'AUC_ROC':   roc_auc_score(y_test_arr[:, i], p),
            'PR_AUC':    average_precision_score(y_test_arr[:, i], p),
        })
    return pd.DataFrame(rows), preds_all, proba_store

results     = {}
pred_store  = {}
proba_store_all = {}
for name, model_dict in trained_models.items():
    df_res, preds, probas = eval_per_label_binary(
        model_dict, test_scaled, y_test, model_thresholds[name], LABELS
    )
    results[name]         = df_res
    pred_store[name]      = preds
    proba_store_all[name] = probas
    print(f'\n--- {name} ---')
    print(df_res.set_index('label').round(4).to_string())


--- RF_base ---
                      F1  Precision  Recall  AUC_ROC  PR_AUC
label                                                       
Anaemia           0.4475     0.4102  0.4923   0.7802  0.4998
Diabetes_Risk     0.4087     0.3670  0.4611   0.8172  0.4327
Dyslipidemia      0.4321     0.3424  0.5858   0.7183  0.4770
Kidney_Risk       0.4497     0.3641  0.5879   0.7521  0.4975
Liver_Stress      0.5628     0.4649  0.7132   0.7947  0.6356
Thyroid_Abnormal  0.5156     0.4321  0.6389   0.7659  0.5724

--- RF_balanced ---
                      F1  Precision  Recall  AUC_ROC  PR_AUC
label                                                       
Anaemia           0.4496     0.3783  0.5540   0.7874  0.4953
Diabetes_Risk     0.4170     0.3473  0.5217   0.8423  0.4435
Dyslipidemia      0.4339     0.3112  0.7168   0.7269  0.4790
Kidney_Risk       0.4460     0.3261  0.7054   0.7561  0.4981
Liver_Stress      0.5639     0.4886  0.6668   0.7962  0.6306
Thyroid_Abnormal  0.5185     0.3932  0.7611   0

In [10]:
# Overall multi-label metrics
print('=== Overall Metrics ===')
overall_rows = []
overall_results = {}  # for FIX E: best model selection
for name, preds in pred_store.items():
    macro_f1 = f1_score(y_test, preds, average='macro', zero_division=0)
    row = {
        'Model':          name,
        'Hamming_Loss':   hamming_loss(y_test, preds),
        'Subset_Accuracy':accuracy_score(y_test, preds),
        'Micro_F1':       f1_score(y_test, preds, average='micro', zero_division=0),
        'Macro_F1':       macro_f1,
    }
    overall_rows.append(row)
    overall_results[name] = {'Macro_F1': macro_f1}

overall_df = pd.DataFrame(overall_rows).set_index('Model')
print(overall_df.round(4).to_string())

=== Overall Metrics ===
             Hamming_Loss  Subset_Accuracy  Micro_F1  Macro_F1
Model                                                         
RF_base            0.2454           0.3047    0.4845    0.4694
RF_balanced        0.2772           0.2326    0.4817    0.4715
KNN                0.4313           0.0527    0.3946    0.3756
SVM                0.3509           0.1047    0.4066    0.3872


## Confusion Matrices (Best Model by Macro F1)

In [11]:
# FIX E: select best model dynamically by macro F1
best_model_name = max(overall_results, key=lambda m: overall_results[m]['Macro_F1'])
print(f'Best model: {best_model_name}  '
      f'(Macro F1 = {overall_results[best_model_name]["Macro_F1"]:.4f})')

preds_best = pred_store[best_model_name]

fig, axes = plt.subplots(2, 3, figsize=(15, 9))
for i, lbl in enumerate(LABELS):
    ax = axes[i // 3][i % 3]
    cm = confusion_matrix(y_test[:, i], preds_best[:, i])
    ConfusionMatrixDisplay(cm, display_labels=['Neg','Pos']).plot(ax=ax, colorbar=False)
    ax.set_title(lbl)
plt.suptitle(f'Confusion Matrices — {best_model_name}', fontsize=14)
plt.tight_layout()
plt.savefig(f'{PLOTS}/04_confusion_matrices.png', bbox_inches='tight')
plt.show()

Best model: RF_balanced  (Macro F1 = 0.4715)


## Precision-Recall Curves per Label

In [12]:
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
for i, lbl in enumerate(LABELS):
    ax = axes[i // 3][i % 3]
    for name, probas in proba_store_all.items():
        p = probas[lbl]
        prec, rec, _ = precision_recall_curve(y_test[:, i], p)
        ax.plot(rec, prec, label=name, alpha=0.8)
    ax.set_title(lbl); ax.set_xlabel('Recall'); ax.set_ylabel('Precision')
    ax.legend(fontsize=7)
plt.suptitle('Precision-Recall Curves per Label', fontsize=14)
plt.tight_layout()
plt.savefig(f'{PLOTS}/04_pr_curves.png', bbox_inches='tight')
plt.show()

In [13]:
all_res = pd.concat(
    [df.assign(Model=name) for name, df in results.items()]
)[['Model','label','F1','Precision','Recall','AUC_ROC','PR_AUC']]
all_res.to_csv(f'{METRICS}/04_classification_results.csv', index=False)

overall_df.reset_index().to_csv(f'{METRICS}/04_overall_metrics.csv', index=False)
print('Metrics saved to results/metrics/')
print(all_res.round(4).to_string())

Metrics saved to results/metrics/
         Model             label      F1  Precision  Recall  AUC_ROC  PR_AUC
0      RF_base           Anaemia  0.4475     0.4102  0.4923   0.7802  0.4998
1      RF_base     Diabetes_Risk  0.4087     0.3670  0.4611   0.8172  0.4327
2      RF_base      Dyslipidemia  0.4321     0.3424  0.5858   0.7183  0.4770
3      RF_base       Kidney_Risk  0.4497     0.3641  0.5879   0.7521  0.4975
4      RF_base      Liver_Stress  0.5628     0.4649  0.7132   0.7947  0.6356
5      RF_base  Thyroid_Abnormal  0.5156     0.4321  0.6389   0.7659  0.5724
0  RF_balanced           Anaemia  0.4496     0.3783  0.5540   0.7874  0.4953
1  RF_balanced     Diabetes_Risk  0.4170     0.3473  0.5217   0.8423  0.4435
2  RF_balanced      Dyslipidemia  0.4339     0.3112  0.7168   0.7269  0.4790
3  RF_balanced       Kidney_Risk  0.4460     0.3261  0.7054   0.7561  0.4981
4  RF_balanced      Liver_Stress  0.5639     0.4886  0.6668   0.7962  0.6306
5  RF_balanced  Thyroid_Abnormal  0.5185  